<a href="https://colab.research.google.com/github/Pes2ug23am092/AIS_transit_procurement_data_analysis/blob/main/Encephalon_warcausalinferenceOLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 1: SETUP AND LOAD DATASETS
# ---------------------------------------------------------------------------------------------

import pandas as pd
from google.colab import drive

# 1. Mount Google Drive (Required for fresh Colab session)
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 2: RE-INSPECT PROCUREMENT DATA TO FIND THE CORRECT DATE COLUMN
# ---------------------------------------------------------------------------------------------

import pandas as pd
from google.colab import drive

# Mount Google Drive (if not already mounted)
try:
    drive.mount('/content/drive')
except:
    pass

# --- Define File Path ---
# Assuming this path is correct based on your previous input.
PROCUREMENT_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/Procurement_Data/processed/final_awards_data/Combined/combined_processed_procurement_data.csv'


print("="*60)
print("--- 🏛️ Procurement Contracts Data Re-Inspection ---")

try:
    # Load the data (Note: The DtypeWarning is common with mixed CSV types, we'll ignore it for now)
    df_procurement = pd.read_csv(PROCUREMENT_DATA_PATH)

    print(f"Procurement Data Shape (Rows, Columns): {df_procurement.shape}")

    # **CRITICAL STEP: Print all column names**
    print("\n✅ Procurement Data Columns (Possible Merge Keys):")
    print(list(df_procurement.columns))

    print("\nProcurement Data Head (First 5 Rows - RAW):")
    # We skip the pd.to_datetime conversion to avoid the previous error
    print(df_procurement.head().to_markdown(index=False))

    print("\nProcurement Data Info (Data Types):")
    df_procurement.info()

except FileNotFoundError:
    print(f"\n❌ ERROR: Procurement file not found at: {PROCUREMENT_DATA_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 🏛️ Procurement Contracts Data Re-Inspection ---
Procurement Data Shape (Rows, Columns): (65884, 17)

✅ Procurement Data Columns (Possible Merge Keys):
['NoticeId', 'Title', 'Department/Ind.Agency', 'Sub-Tier', 'PostedDate', 'Type', 'BaseType', 'ArchiveDate', 'SetASide', 'NaicsCode', 'ClassificationCode', 'PopCity', 'PopState', 'AwardDate', 'Award$', 'Awardee', 'Description']

Procurement Data Head (First 5 Rows - RAW):
| NoticeId                         | Title                                                                        | Department/Ind.Agency   | Sub-Tier         | PostedDate             | Type         | BaseType                       | ArchiveDate   |   SetASide |   NaicsCode | ClassificationCode   | PopCity    | PopState   | AwardDate   | Award$        | Awardee                                                               | Description     

/tmp/ipython-input-3832096062.py:24: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_procurement = pd.read_csv(PROCUREMENT_DATA_PATH)


In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 3 & 4: AGGREGATE, MERGE, AND SAVE FUSED DATA (FIXED TO LEFT JOIN)
# ---------------------------------------------------------------------------------------------

import pandas as pd
import numpy as np
from google.colab import drive
import os

# --- Configuration (Based on Previous Steps) ---
AIS_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/monthly_trade_analysis_ITS_ready.csv'
PROCUREMENT_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/Procurement_Data/processed/final_awards_data/Combined/combined_processed_procurement_data.csv'

# Ensure Drive is mounted
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# --- 1. Load Data ---
print("Reloading AIS and Procurement Data...")
df_ais = pd.read_csv(AIS_DATA_PATH)
df_procurement = pd.read_csv(PROCUREMENT_DATA_PATH, dtype={'Award$': str})


# --- 2. Clean and Aggregate Procurement Data ---
print("--- 🏛️ Aggregating Procurement Data to Monthly Time Series ---")

# A. CLEAN AWARD VALUE
df_procurement['Contract_Value_Numeric'] = (
    df_procurement['Award$']
    .astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .str.extract(r'(\d+\.?\d*)', expand=False)
    .astype(float)
)

# B. FIX AWARD DATE AND AGGREGATE
df_procurement['AwardDate'] = pd.to_datetime(
    df_procurement['AwardDate'],
    utc=True,
    errors='coerce' # Handles out-of-bounds dates
).dt.tz_localize(None)


# Aggregate: Group by month and sum the contract values, count the notices.
df_procurement_agg = df_procurement.groupby(
    pd.to_datetime(df_procurement['AwardDate']).dt.to_period('M').dt.to_timestamp()
).agg(
    Total_Contract_Value=('Contract_Value_Numeric', 'sum'),
    Count_of_Contracts=('NoticeId', 'count')
).reset_index()

# Standardize the merge key name
df_procurement_agg = df_procurement_agg.rename(columns={'AwardDate': 'Time_Period'})
print(f"✅ Procurement Aggregation Complete. Shape: {df_procurement_agg.shape}")


# --- 3. Prepare for Merge ---
df_ais['Time_Period'] = pd.to_datetime(df_ais['Time_Period']).dt.normalize()
df_procurement_agg['Time_Period'] = df_procurement_agg['Time_Period'].dt.normalize()


# --- 4. Merge Data (CRITICAL FIX: LEFT JOIN) ---
df_fused = pd.merge(
    df_ais,
    df_procurement_agg,
    on='Time_Period',
    how='left' # KEEP ALL 60 MONTHS OF AIS DATA
)

print("\n--- 🧩 Fused Data Inspection ---")
print(f"✅ Fused Data Shape: {df_fused.shape} (NOW 60 ROWS)")
print("Fused Data Head (Showing AIS and Procurement Metrics):")
print(df_fused[['Time_Period', 'Average_Draft_Meters', 'Total_Contract_Value', 'Intervention_D']].head().to_markdown(index=False))

# Check for NaNs created by the merge (should be 6 NaNs)
na_count = df_fused[['Total_Contract_Value', 'Count_of_Contracts']].isnull().sum()
print(f"\nNaNs introduced by Left Merge:\n{na_count.to_markdown()}")


# --- 5. Clean NaNs and Save the Final Output File ---
# Replace NaNs with 0 for the ITS regression model (missing contract month = $0 value)
df_fused['Total_Contract_Value'] = df_fused['Total_Contract_Value'].fillna(0)
df_fused['Count_of_Contracts'] = df_fused['Count_of_Contracts'].fillna(0)

FINAL_FUSED_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/war_final/fused_supply_chain_data_ITS_ready.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(FINAL_FUSED_DATA_PATH), exist_ok=True)

df_fused.to_csv(FINAL_FUSED_DATA_PATH, index=False)
print(f"\n🚀 Final ITS-ready Fused data successfully saved to: {FINAL_FUSED_DATA_PATH}")

Mounted at /content/drive
Reloading AIS and Procurement Data...
--- 🏛️ Aggregating Procurement Data to Monthly Time Series ---
✅ Procurement Aggregation Complete. Shape: (113, 3)

--- 🧩 Fused Data Inspection ---
✅ Fused Data Shape: (60, 11) (NOW 60 ROWS)
Fused Data Head (Showing AIS and Procurement Metrics):
| Time_Period         |   Average_Draft_Meters |   Total_Contract_Value |   Intervention_D |
|:--------------------|-----------------------:|-----------------------:|-----------------:|
| 2020-01-01 00:00:00 |               0.50606  |            3.64646e+10 |                0 |
| 2020-02-01 00:00:00 |               0.529248 |            2.73591e+10 |                0 |
| 2020-03-01 00:00:00 |               0.640795 |            2.57252e+10 |                0 |
| 2020-04-01 00:00:00 |               0.95723  |            2.85568e+10 |                0 |
| 2020-05-01 00:00:00 |               1.06365  |            5.88634e+10 |                0 |

NaNs introduced by Left Merge:
|      

In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 5: CAUSAL INFERENCE - INTERRUPTED TIME SERIES (ITS) REGRESSION
# ---------------------------------------------------------------------------------------------

import pandas as pd
import statsmodels.api as sm

# --- 1. Define File Path ---
FINAL_FUSED_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/war_final/fused_supply_chain_data_ITS_ready.csv'

# --- 2. Load the Final Fused Data ---
df = pd.read_csv(FINAL_FUSED_DATA_PATH)

# --- 3. Define the ITS Model Formula ---
# The formula specifies the relationship:
# Dependent Variable ~ Independent Variables
# The 'C' indicates that Total_Contract_Value is being used as a control variable.
formula = 'Average_Draft_Meters ~ Time + Intervention_D + Post_Slope_P + Total_Contract_Value'

# --- 4. Run the OLS Regression ---
# Add a constant (intercept) to the model
df = sm.add_constant(df)

# Initialize and fit the model
model = sm.formula.ols(formula, data=df)
results = model.fit()

# --- 5. Print Results ---
print("\n" + "="*80)
print("--- 🔬 CAUSAL INFERENCE RESULTS: ITS MODEL WITH PROCUREMENT CONTROL ---")
print("Dependent Variable: Average_Draft_Meters")
print("Intervention: Russia-Ukraine War (March 2022)")
print("="*80)
print(results.summary())

# Save the full results summary to a text file for documentation
results_summary = results.summary().as_text()
summary_path = '/content/drive/MyDrive/ADA Encephalon/war_final/causal_inference_summary.txt'
with open(summary_path, 'w') as f:
    f.write(results_summary)
print(f"\n✅ Regression summary saved to: {summary_path}")


--- 🔬 CAUSAL INFERENCE RESULTS: ITS MODEL WITH PROCUREMENT CONTROL ---
Dependent Variable: Average_Draft_Meters
Intervention: Russia-Ukraine War (March 2022)
                             OLS Regression Results                             
Dep. Variable:     Average_Draft_Meters   R-squared:                       0.661
Model:                              OLS   Adj. R-squared:                  0.636
Method:                   Least Squares   F-statistic:                     26.79
Date:                  Sat, 15 Nov 2025   Prob (F-statistic):           2.34e-12
Time:                          17:36:26   Log-Likelihood:                 45.281
No. Observations:                    60   AIC:                            -80.56
Df Residuals:                        55   BIC:                            -70.09
Df Model:                             4                                         
Covariance Type:              nonrobust                                         
                           coef

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# --- Configuration ---
FINAL_FUSED_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/war_final/fused_supply_chain_data_ITS_ready.csv'
INTERVENTION_MONTH_INDEX = 26 # The war starts in March 2022, which is Time=26 (starting from 0)

# --- 1. Load Data ---
df = pd.read_csv(FINAL_FUSED_DATA_PATH)
df['Time_Period'] = pd.to_datetime(df['Time_Period'])

# --- 2. Re-Run OLS Model to Get Coefficients ---
# Add a constant and run the model again (necessary for extracting results)
df = sm.add_constant(df, prepend=False)
formula = 'Average_Draft_Meters ~ Time + Intervention_D + Post_Slope_P + Total_Contract_Value + const'
results = sm.formula.ols(formula, data=df).fit()

# Extract Coefficients (beta values)
B0 = results.params['const']
B1 = results.params['Time']
B2 = results.params['Intervention_D']
B3 = results.params['Post_Slope_P']
B4 = results.params['Total_Contract_Value']


# --- 3. Calculate Counterfactual and Predicted Trends ---

# B. Counterfactual Trend (What would have happened without the war)
# This is B0 + B1*Time + B4*Total_Contract_Value (i.e., setting D and P to zero)
df['Counterfactual_Y'] = B0 + B1 * df['Time'] + B4 * df['Total_Contract_Value']

# C. Pre-War Trend (for plotting the segmented line)
df['Pre_War_Trend'] = df['Counterfactual_Y'].copy()
df.loc[df['Time'] >= INTERVENTION_MONTH_INDEX, 'Pre_War_Trend'] = np.nan

# D. Post-War Trend (for plotting the segmented line)
# This uses the full model: B0 + B1*Time + B2*D + B3*P + B4*ContractValue
df['Post_War_Trend'] = df['Counterfactual_Y'] + B2 * df['Intervention_D'] + B3 * df['Post_Slope_P']
df.loc[df['Time'] < INTERVENTION_MONTH_INDEX, 'Post_War_Trend'] = np.nan

# --- 4. Plot 1: ITS Causal Inference Plot ---
plt.figure(figsize=(12, 6))

# Plot the Actual Data
plt.plot(df['Time_Period'], df['Average_Draft_Meters'], 'o-', color='black', label='Actual Average Vessel Draft', markersize=4)

# Plot the Segmented Trend Line (Counterfactual and Post-Intervention)
plt.plot(df['Time_Period'], df['Pre_War_Trend'], '--', color='blue', label='Pre-War Trend (Counterfactual)', linewidth=2)
plt.plot(df['Time_Period'], df['Post_War_Trend'], '-', color='red', label='Post-War Trend (Causal Effect)', linewidth=2)


# Add Intervention Line
plt.axvline(df['Time_Period'][INTERVENTION_MONTH_INDEX], color='gray', linestyle=':', linewidth=2, label='War Start (Mar 2022)')

plt.title('Causal Effect of Russia-Ukraine War on Shipping Volume (Vessel Draft)')
plt.xlabel('Time (Monthly)')
plt.ylabel('Average Vessel Draft (Meters)')
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('its_causal_inference_plot.png')
plt.close()

# --- 5. Plot 2: Procurement Control Variable Time Series ---
plt.figure(figsize=(12, 4))
# Convert to Billions for readability
plt.plot(df['Time_Period'], df['Total_Contract_Value'] / 1e9, color='green', label='Total Contract Value (USD Billions)')
plt.axvline(df['Time_Period'][INTERVENTION_MONTH_INDEX], color='gray', linestyle=':', linewidth=2, label='War Start (Mar 2022)')
plt.title('Procurement Control Variable Time Series')
plt.xlabel('Time (Monthly)')
plt.ylabel('Total Contract Value (USD Billions)')
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('procurement_control_plot.png')
plt.close()

print("✅ Two presentation-ready graphs generated: its_causal_inference_plot.png and procurement_control_plot.png")

✅ Two presentation-ready graphs generated: its_causal_inference_plot.png and procurement_control_plot.png


In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 7: DUAL-AXIS PLOT FOR CORRELATION VISUALIZATION
# ---------------------------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# --- Configuration ---
FINAL_FUSED_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/war_final/fused_supply_chain_data_ITS_ready.csv'
INTERVENTION_MONTH_INDEX = 26

# --- 1. Load Data ---
df = pd.read_csv(FINAL_FUSED_DATA_PATH)
df['Time_Period'] = pd.to_datetime(df['Time_Period'])


# --- 2. Create Dual-Axis Plot ---
fig, ax1 = plt.subplots(figsize=(12, 6))

# Axis 1: Average Vessel Draft (Meters)
color = 'tab:blue'
ax1.set_xlabel('Time (Monthly)')
ax1.set_ylabel('Avg. Vessel Draft (Meters)', color=color)
ax1.plot(df['Time_Period'], df['Average_Draft_Meters'], color=color, linewidth=2, label='AIS: Vessel Draft')
ax1.tick_params(axis='y', labelcolor=color)

# Create Secondary Axis (Twin Axis)
ax2 = ax1.twinx()

# Axis 2: Total Contract Value (USD Billions)
color = 'tab:green'
ax2.set_ylabel('Total Contract Value (USD Billions)', color=color)
ax2.plot(df['Time_Period'], df['Total_Contract_Value'] / 1e9, color=color, linestyle='--', alpha=0.6, label='Procurement: Contract Value')
ax2.tick_params(axis='y', labelcolor=color)

# Add Intervention Line
plt.axvline(df['Time_Period'][INTERVENTION_MONTH_INDEX], color='gray', linestyle=':', linewidth=2, label='War Start')

plt.title('Time Series Comparison: Vessel Draft vs. Procurement Spending')
fig.tight_layout()
plt.legend(loc='upper left')
plt.grid(True, which='major', axis='x', linestyle='--')
plt.savefig('dual_axis_trend_comparison.png')
plt.close()

print("✅ Dual-axis correlation plot generated: dual_axis_trend_comparison.png")

✅ Dual-axis correlation plot generated: dual_axis_trend_comparison.png


In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 8: CALCULATE PEARSON'S R CORRELATION
# ---------------------------------------------------------------------------------------------
import numpy as np

# Calculate the correlation coefficient (R)
correlation = df['Average_Draft_Meters'].corr(df['Total_Contract_Value'])

# Calculate the R-squared value
r_squared = correlation**2

print(f"\n--- Statistical Correlation (Pearson's R) ---")
print(f"R (Correlation Coefficient): {correlation:.3f}")
print(f"R-squared: {r_squared:.3f} (Only {r_squared*100:.1f}% of Draft variance is explained by Procurement spending)")


--- Statistical Correlation (Pearson's R) ---
R (Correlation Coefficient): -0.018
R-squared: 0.000 (Only 0.0% of Draft variance is explained by Procurement spending)


In [ ]:
# ---------------------------------------------------------------------------------------------
# STEP 9: CALCULATE AND VISUALIZE LAGGED CORRELATION (CROSS-CORRELATION)
# ---------------------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
FINAL_FUSED_DATA_PATH = '/content/drive/MyDrive/ADA Encephalon/war_final/fused_supply_chain_data_ITS_ready.csv'

# --- 1. Load Data ---
df = pd.read_csv(FINAL_FUSED_DATA_PATH)
df['Time_Period'] = pd.to_datetime(df['Time_Period'])

# --- 2. Calculate Cross-Correlation for Lags 1 to 12 ---
# We look for a positive correlation: Procurement today leads to shipping later.

max_lag = 12
correlations = {}

for lag in range(1, max_lag + 1):
    # Shift the procurement data backward by 'lag' months
    # E.g., lag=1: correlates Procurement(t-1) with Draft(t)
    lagged_procurement = df['Total_Contract_Value'].shift(lag)

    # Calculate Pearson's R, ignoring NaN values created by shifting
    correlation = df['Average_Draft_Meters'].corr(lagged_procurement)
    correlations[lag] = correlation

# Convert results to DataFrame for easy plotting
df_corr = pd.DataFrame(correlations.items(), columns=['Lag (Months)', 'Correlation (R)'])

print("✅ Lagged Correlation Calculation Complete.")
print(df_corr.to_markdown(index=False))

# --- 3. Visualize Lagged Correlation ---
plt.figure(figsize=(10, 5))
plt.bar(df_corr['Lag (Months)'], df_corr['Correlation (R)'], color='skyblue')

# Highlight the significant line
plt.axhline(0.3, color='red', linestyle='--', linewidth=1, label='Significance Threshold (R=0.3)')
plt.axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.title('Lagged Correlation: Procurement Spending $\\to$ Vessel Draft')
plt.xlabel('Lag (Months): Procurement (t-Lag) vs. Draft (t)')
plt.ylabel("Pearson's Correlation Coefficient (R)")
plt.xticks(df_corr['Lag (Months)'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('lagged_correlation_plot.png')
plt.close()

print("✅ Lagged correlation plot generated: lagged_correlation_plot.png")

✅ Lagged Correlation Calculation Complete.
|   Lag (Months) |   Correlation (R) |
|---------------:|------------------:|
|              1 |         0.0144833 |
|              2 |         0.0151021 |
|              3 |         0.0359692 |
|              4 |         0.0400408 |
|              5 |         0.0283825 |
|              6 |         0.0813932 |
|              7 |         0.03001   |
|              8 |         0.052238  |
|              9 |         0.0295712 |
|             10 |        -0.022512  |
|             11 |        -0.0144512 |
|             12 |         0.0260303 |
✅ Lagged correlation plot generated: lagged_correlation_plot.png
